# HAGSS: Hierarchical Adaptive Group Self-Support Learning with Boundary-Aware Calibration (Kaggle T4x2 Optimized)

This notebook contains a complete, self-contained implementation of the HAGSS framework for incomplete multi-modal brain tumor segmentation, optimized for Kaggle dual Tesla T4 GPUs (T4x2).

---

## 1. Imports and Environment Setup
We import standard libraries, PyTorch modules, NumPy, and SciPy for boundary detection and evaluation metrics. We also check for GPU availability.

In [ ]:
import os
import sys
import math
import time
import json
import random
import logging
import argparse
import platform
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from scipy import ndimage

# Check PyTorch version and device compatibility
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")


---

## 2. HAGSS Configuration
Defines the default hyperparameters matching the paper exactly.
* **Sensitivity Priors:** Defines class-to-modality sensitivity mapping. ED (Edema) uses a double-leader configuration (`FLAIR` + `T2`).
* **HAGF thresholds:** `theta_low` (0.4) and `theta_high` (0.7) for adaptive override, alongside GSS+ voting thresholds.
* **BAC parameters:** `lambda_b` (2.0) and `lambda_u` (1.0) control the intensity of boundary and uncertainty calibration.

In [ ]:
@dataclass
class HAGSSConfig:
    """Central configuration for the HAGSS framework."""

    # Modality & class definitions
    num_modalities: int = 4          # FLAIR, T1, T1c, T2
    num_classes: int = 4             # BG, NCR/NET, ED, ET
    modality_names: Tuple[str, ...] = ("FLAIR", "T1", "T1c", "T2")
    class_names: Tuple[str, ...] = ("BG", "NCR_NET", "ED", "ET")

    # Fixed sensitivity priors
    # Maps class_index -> list of leader modality indices
    sensitivity_priors: Dict[int, List[int]] = field(default_factory=lambda: {
        0: [0],     # BG      -> FLAIR
        1: [2],     # NCR/NET -> T1c
        2: [0, 3],  # ED      -> FLAIR + T2  (double-leader)
        3: [2],     # ET      -> T1c
    })

    # Logit standardization
    standardize_eps: float = 1e-6  # numerical stability for std

    # Hierarchical Adaptive Group Formation
    theta_low: float = 0.4    # low confidence threshold for override
    theta_high: float = 0.7   # high confidence threshold for override

    # GSS+ voting thresholds
    leader_threshold: float = 0.65   # T_L — accept leader logit if above
    member_threshold: float = 0.75   # T_M — members override if unanimous
    penalty_coeff: float = 0.3       # rho — penalty for uncertain fallback

    # Boundary-Aware Calibration
    lambda_b: float = 2.0     # weight for ground-truth boundary term
    lambda_u: float = 1.0     # weight for prediction uncertainty term
    sobel_threshold: float = 0.5  # gradient magnitude threshold for B_gt

    # Cross-Scale Consistency
    downsample_factor: int = 2  # spatial downsample ratio

    # Total loss weights
    alpha: float = 0.7        # task loss weight
    beta: float = 0.3         # boundary-aware KD loss weight
    gamma: float = 0.1        # cross-scale consistency weight
    tau: float = 8.0          # distillation temperature

    # Reliability filtering
    initial_unreliable_ratio: float = 0.08  # alpha_0 = 8%
    mask_rate: float = 0.2                  # random dropout rate (20%)

    # Training schedule
    total_epochs: int = 1200
    reload_interval: int = 300   # reload model every N epochs


---

## 3. Logit Standardization
Standardizes each modality's logit map to zero mean and unit variance per voxel across the class dimension. This normalizes logits from heterogeneous encoders before grouping.

In [ ]:
class LogitStandardization(nn.Module):
    """Per-voxel zero-mean, unit-variance standardisation of logits."""

    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        # Compute mean and std across the class dimension (dim=1)
        mean = logits.mean(dim=1, keepdim=True)             # (B,1,D,H,W)
        std = logits.std(dim=1, keepdim=True) + self.eps     # (B,1,D,H,W)
        return (logits - mean) / std


---

## 4. Entropy & Confidence Scoring
Computes prediction entropy $H_m(v)$ and normalized confidence $C_m(v) \in [0, 1]$ per voxel. A value of $1$ represents full confidence.

In [ ]:
class EntropyConfidence(nn.Module):
    """Compute per-voxel prediction entropy and normalised confidence."""

    def __init__(self, num_classes: int = 4, eps: float = 1e-8):
        super().__init__()
        self.num_classes = num_classes
        self.eps = eps
        self.log_n = math.log(num_classes)

    def forward(self, logits: torch.Tensor):
        probs = F.softmax(logits, dim=1)                       # (B,C,D,H,W)
        log_probs = torch.log(probs + self.eps)
        entropy = -(probs * log_probs).sum(dim=1)              # (B,D,H,W)
        confidence = 1.0 - entropy / self.log_n
        confidence = confidence.clamp(0.0, 1.0)
        return entropy, confidence


---

## 5. 3D Sobel Boundary Detection
Applies 3D Sobel filtering along the depth, height, and width axes of the one-hot encoded ground-truth label map to construct a binary boundary mask $B_{gt}(v) \in \{0, 1\}$.

In [ ]:
def _build_sobel_kernels_3d() -> torch.Tensor:
    """Build three 3×3×3 Sobel kernels for the D, H, W axes."""
    s = torch.tensor([1.0, 2.0, 1.0])
    d = torch.tensor([-1.0, 0.0, 1.0])

    k_d = d.view(3, 1, 1) * s.view(1, 3, 1) * s.view(1, 1, 3)
    k_h = s.view(3, 1, 1) * d.view(1, 3, 1) * s.view(1, 1, 3)
    k_w = s.view(3, 1, 1) * s.view(1, 3, 1) * d.view(1, 1, 3)

    kernels = torch.stack([k_d, k_h, k_w], dim=0)
    return kernels.unsqueeze(1)                       # (3, 1, 3, 3, 3)


class SobelBoundary3D(nn.Module):
    """Detect ground-truth boundaries using 3D Sobel filtering."""

    def __init__(self, num_classes: int = 4, threshold: float = 0.5):
        super().__init__()
        self.num_classes = num_classes
        self.threshold = threshold
        self.register_buffer("kernels", _build_sobel_kernels_3d())

    def forward(self, labels: torch.Tensor) -> torch.Tensor:
        B = labels.shape[0]
        device = labels.device

        # One-hot encode: (B, C, D, H, W)
        one_hot = F.one_hot(labels.long(), self.num_classes)  # (B,D,H,W,C)
        one_hot = one_hot.permute(0, 4, 1, 2, 3).float()     # (B,C,D,H,W)

        # Accumulate gradient magnitude over all classes
        grad_mag_sq = torch.zeros(B, 1, *labels.shape[1:], device=device)
        kernels = self.kernels

        for c in range(self.num_classes):
            channel = one_hot[:, c:c+1, :, :, :]
            # Replicate padding to avoid false borders
            padded = F.pad(channel, (1,1,1,1,1,1), mode="replicate")
            grads = F.conv3d(padded, kernels, padding=0)  # (B, 3, D, H, W)
            grad_mag_sq += (grads ** 2).sum(dim=1, keepdim=True)

        grad_mag = torch.sqrt(grad_mag_sq + 1e-8)
        boundary = (grad_mag > self.threshold).float()
        return boundary


---

## 6. Boundary-Aware Calibration (BAC)
Computes the spatial calibration weight map:
$$W(v) = 1 + \lambda_b \cdot B_{gt}(v) + \lambda_u \cdot U(v)$$
and evaluates the boundary-aware KL divergence distillation loss.

In [ ]:
class BoundaryAwareCalibration(nn.Module):
    """Compute spatially-weighted distillation loss."""

    def __init__(
        self,
        num_classes: int = 4,
        lambda_b: float = 2.0,
        lambda_u: float = 1.0,
        sobel_threshold: float = 0.5,
        tau: float = 8.0,
    ):
        super().__init__()
        self.lambda_b = lambda_b
        self.lambda_u = lambda_u
        self.tau = tau

        self.sobel = SobelBoundary3D(num_classes=num_classes, threshold=sobel_threshold)
        self.entropy_conf = EntropyConfidence(num_classes=num_classes)

    def compute_weight_map(
        self,
        labels: torch.Tensor,
        logits_list: List[torch.Tensor],
    ) -> torch.Tensor:
        B_gt = self.sobel(labels)

        # Compute average normalised entropy across modalities
        entropies = []
        for logits in logits_list:
            ent, _ = self.entropy_conf(logits)
            ent_norm = ent / self.entropy_conf.log_n
            entropies.append(ent_norm)
        U = torch.stack(entropies, dim=0).mean(dim=0)     # (B, D, H, W)
        U = U.unsqueeze(1)                                  # (B, 1, D, H, W)

        W = 1.0 + self.lambda_b * B_gt + self.lambda_u * U
        return W

    def forward(
        self,
        student_logits: torch.Tensor,
        pseudo_target: torch.Tensor,
        weight_map: torch.Tensor,
        reliability_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        student_log_p = F.log_softmax(student_logits / self.tau, dim=1)
        target_p = F.softmax(pseudo_target / self.tau, dim=1).detach()

        kl = F.kl_div(student_log_p, target_p, reduction="none").sum(dim=1)
        kl = kl.unsqueeze(1)                              # (B, 1, D, H, W)

        weighted_kl = weight_map * kl

        if reliability_mask is not None:
            weighted_kl = weighted_kl * reliability_mask

        return weighted_kl.mean()


---

## 7. Hierarchical Adaptive Group Formation (HAGF)
Adapts GSS+ voting per voxel. It dynamically overrides the fixed default modality leader path when:
1. The default leader confidence falls below `theta_low`.
2. An alternative available modality confidence exceeds `theta_high`.

In [ ]:
class HierarchicalAdaptiveGroupFormation(nn.Module):
    """Constructs pseudo-target logits via adaptive leader selection + voting."""

    def __init__(
        self,
        num_classes: int = 4,
        num_modalities: int = 4,
        sensitivity_priors: Dict[int, List[int]] = None,
        theta_low: float = 0.4,
        theta_high: float = 0.7,
        leader_threshold: float = 0.65,
        member_threshold: float = 0.75,
        penalty_coeff: float = 0.3,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.num_modalities = num_modalities
        self.theta_low = theta_low
        self.theta_high = theta_high
        self.T_L = leader_threshold
        self.T_M = member_threshold
        self.rho = penalty_coeff

        if sensitivity_priors is None:
            sensitivity_priors = {
                0: [0],     # BG      -> FLAIR
                1: [2],     # NCR/NET -> T1c
                2: [0, 3],  # ED      -> FLAIR, T2
                3: [2],     # ET      -> T1c
            }
        self.priors = sensitivity_priors

        self.standardize = LogitStandardization()
        self.entropy_conf = EntropyConfidence(num_classes=num_classes)

    def _get_leader_logit(
        self,
        std_logits: torch.Tensor,
        confidences: torch.Tensor,
        class_idx: int,
    ) -> torch.Tensor:
        prior_leaders = self.priors[class_idx]
        M = std_logits.shape[0]

        # Get default leader signal
        if len(prior_leaders) == 1:
            default_logit = std_logits[prior_leaders[0], :, class_idx]
            default_conf  = confidences[prior_leaders[0]]
        else:
            # ED double-leader average
            leader_logits = torch.stack([std_logits[m, :, class_idx] for m in prior_leaders], dim=0)
            default_logit = leader_logits.mean(dim=0)
            leader_confs = torch.stack([confidences[m] for m in prior_leaders], dim=0)
            default_conf = leader_confs.mean(dim=0)

        # Get best alternative modality
        all_confs = confidences
        best_alt_conf, best_alt_idx = all_confs.max(dim=0)

        # Adaptive override mask
        override_mask = ((default_conf < self.theta_low) & (best_alt_conf > self.theta_high)).float()

        alt_logit = torch.zeros_like(default_logit)
        for m in range(M):
            m_mask = (best_alt_idx == m).float()
            alt_logit += m_mask * std_logits[m, :, class_idx]

        leader_logit = (1 - override_mask) * default_logit + override_mask * alt_logit
        return leader_logit

    def _vote_single_class(
        self,
        leader_logit: torch.Tensor,
        member_logits: torch.Tensor,
    ) -> torch.Tensor:
        M = member_logits.shape[0]

        # 1. Leader accepted
        leader_accept = (leader_logit > self.T_L).float()

        # 2. Member unanimous override
        member_above = (member_logits > self.T_M).float()
        unanimous = (member_above.sum(dim=0) == M).float()

        # 3. Fallback to min logit with penalty
        min_logit, _ = member_logits.min(dim=0)
        fallback = min_logit - self.rho

        voted = torch.where(
            leader_accept.bool(),
            leader_logit,
            torch.where(
                unanimous.bool(),
                member_logits.mean(dim=0),
                fallback,
            ),
        )
        return voted

    def forward(self, logits_list: List[torch.Tensor]) -> torch.Tensor:
        M = len(logits_list)
        B, C = logits_list[0].shape[:2]

        # Standardise logits
        std_logits = torch.stack([self.standardize(lg) for lg in logits_list], dim=0)

        # Confidence scores from raw logits
        confidences = []
        for m in range(M):
            _, conf = self.entropy_conf(logits_list[m])
            confidences.append(conf)
        confidences = torch.stack(confidences, dim=0)

        voted_channels = []
        for c in range(C):
            leader_logit = self._get_leader_logit(std_logits, confidences, c)
            member_logits = std_logits[:, :, c]
            voted = self._vote_single_class(leader_logit, member_logits)
            voted_channels.append(voted)

        pseudo_target = torch.stack(voted_channels, dim=1)
        return pseudo_target


---

## 8. Cross-Scale Consistency (CSC)
Maintains spatial smoothness. It average-pools the logits by a factor of 2, evaluates HAGF at the lower scale, upsamples it, and penalizes discrepancy via KL divergence.

In [ ]:
class CrossScaleConsistency(nn.Module):
    """KL consistency between full- and half-resolution pseudo-targets."""

    def __init__(
        self,
        group_formation: HierarchicalAdaptiveGroupFormation,
        downsample_factor: int = 2,
        tau: float = 8.0,
    ):
        super().__init__()
        self.group_formation = group_formation
        self.factor = downsample_factor
        self.tau = tau

    def forward(
        self,
        logits_list: List[torch.Tensor],
        pseudo_target_full: torch.Tensor,
        reliability_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        f = self.factor
        original_size = logits_list[0].shape[2:]

        # Check dimension divisibility
        for dim_name, dim_val in zip(("D", "H", "W"), original_size):
            assert dim_val % f == 0, (
                f"CSC requires spatial dims divisible by {f}, "
                f"but {dim_name}={dim_val}."
            )

        # Average pool
        down_logits = [F.avg_pool3d(lg, kernel_size=f, stride=f) for lg in logits_list]

        # Group formation on pooled features
        pseudo_down = self.group_formation(down_logits)

        # Upsample back
        pseudo_down_up = F.interpolate(
            pseudo_down,
            size=original_size,
            mode="trilinear",
            align_corners=False,
        )

        log_p_full = F.log_softmax(pseudo_target_full / self.tau, dim=1)
        p_down = F.softmax(pseudo_down_up / self.tau, dim=1).detach()

        kl = F.kl_div(log_p_full, p_down, reduction="none").sum(dim=1)

        if reliability_mask is not None:
            kl = kl * reliability_mask.squeeze(1)

        return kl.mean()


---

## 9. Reliability Filtering
Decays the fraction of unreliable voxels excluded from distillation losses linearly from $\alpha_0$ to 0 over training epochs:
$$\alpha_t = \alpha_0 \left( 1 - \frac{t}{T} \right)$$

In [ ]:
class ReliabilityFilter(nn.Module):
    """Entropy-based reliability mask with linear decay schedule."""

    def __init__(self, alpha_0: float = 0.08, total_epochs: int = 1200):
        super().__init__()
        self.alpha_0 = alpha_0
        self.total_epochs = total_epochs

    def get_alpha(self, epoch: int) -> float:
        return self.alpha_0 * max(0.0, 1.0 - epoch / self.total_epochs)

    def forward(self, entropy: torch.Tensor, epoch: int) -> torch.Tensor:
        alpha_t = self.get_alpha(entropy, epoch) if hasattr(self, 'get_alpha_sig') else self.get_alpha(epoch)
        if alpha_t <= 0:
            return torch.ones(entropy.shape[0], 1, *entropy.shape[1:], device=entropy.device)

        B = entropy.shape[0]
        flat = entropy.view(B, -1)
        k = max(1, int((1.0 - alpha_t) * flat.shape[1]))
        threshold, _ = flat.kthvalue(k, dim=1)

        threshold = threshold.view(B, *([1] * (entropy.dim() - 1)))
        mask = (entropy <= threshold).float().unsqueeze(1)
        return mask


---

## 10. Modality Masking
Simulates 15 possible incomplete modalities by randomly dropping modalities independently with probability `mask_rate`.

In [ ]:
def generate_modality_mask(
    batch_size: int,
    num_modalities: int = 4,
    mask_rate: float = 0.2,
    device: torch.device = None,
) -> torch.Tensor:
    mask = (torch.rand(batch_size, num_modalities, device=device) >= mask_rate).float()

    # Ensure at least one modality survives per sample
    all_zero = mask.sum(dim=1) == 0
    if all_zero.any():
        rand_idx = torch.randint(0, num_modalities, (all_zero.sum(),), device=device)
        rows = torch.where(all_zero)[0]
        mask[rows, rand_idx] = 1.0

    return mask


---

## 11. Complete HAGSS Loss Component
Aggregates standard per-modality segmentation task cross-entropy, Boundary-Aware Knowledge Distillation (BA-KD) loss, and Cross-Scale Consistency (CSC) loss.

In [ ]:
class HAGSSLoss(nn.Module):
    """Complete HAGSS training loss module."""

    def __init__(self, cfg: HAGSSConfig = None):
        super().__init__()
        if cfg is None:
            cfg = HAGSSConfig()
        self.cfg = cfg

        self.group_formation = HierarchicalAdaptiveGroupFormation(
            num_classes=cfg.num_classes,
            num_modalities=cfg.num_modalities,
            sensitivity_priors=cfg.sensitivity_priors,
            theta_low=cfg.theta_low,
            theta_high=cfg.theta_high,
            leader_threshold=cfg.leader_threshold,
            member_threshold=cfg.member_threshold,
            penalty_coeff=cfg.penalty_coeff,
        )

        self.bac = BoundaryAwareCalibration(
            num_classes=cfg.num_classes,
            lambda_b=cfg.lambda_b,
            lambda_u=cfg.lambda_u,
            sobel_threshold=cfg.sobel_threshold,
            tau=cfg.tau,
        )

        self.csc = CrossScaleConsistency(
            group_formation=self.group_formation,
            downsample_factor=cfg.downsample_factor,
            tau=cfg.tau,
        )

        self.reliability_filter = ReliabilityFilter(
            alpha_0=cfg.initial_unreliable_ratio,
            total_epochs=cfg.total_epochs,
        )

        self.entropy_conf = EntropyConfidence(num_classes=cfg.num_classes)

    @staticmethod
    def _task_loss(
        logits_list: List[torch.Tensor],
        labels: torch.Tensor,
        modality_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        M = len(logits_list)
        B = labels.shape[0]
        num_voxels = labels[0].numel()
        total_loss = torch.tensor(0.0, device=labels.device)
        n_active_modalities = 0

        for m in range(M):
            ce = F.cross_entropy(logits_list[m], labels.long(), reduction="none")

            if modality_mask is not None:
                mask_m = modality_mask[:, m]
                n_active_samples = mask_m.sum().clamp(min=1e-8)
                mask_broad = mask_m.view(B, *([1] * (ce.dim() - 1)))
                loss_m = (ce * mask_broad).sum() / (n_active_samples * num_voxels)
                if mask_m.sum() > 0:
                    total_loss = total_loss + loss_m
                    n_active_modalities += 1
            else:
                total_loss = total_loss + ce.mean()
                n_active_modalities += 1

        return total_loss / max(n_active_modalities, 1)

    def forward(
        self,
        logits_list: List[torch.Tensor],
        labels: torch.Tensor,
        epoch: int,
        modality_mask: torch.Tensor = None,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        cfg = self.cfg

        # 1. Task Loss
        L_task = self._task_loss(logits_list, labels, modality_mask)

        # 2. Reliability Mask
        avg_entropy = torch.zeros_like(labels, dtype=torch.float32)
        for lg in logits_list:
            ent, _ = self.entropy_conf(lg)
            avg_entropy = avg_entropy + ent
        avg_entropy = avg_entropy / len(logits_list)

        reliability_mask = self.reliability_filter(avg_entropy, epoch)

        # 3. Pseudo-Target
        pseudo_target = self.group_formation(logits_list)

        # 4. Weight Map
        weight_map = self.bac.compute_weight_map(labels, logits_list)

        # 5. BAC distillation
        L_bac_kd = torch.tensor(0.0, device=labels.device)
        n_modalities = len(logits_list)
        for m in range(n_modalities):
            loss_m = self.bac(
                student_logits=logits_list[m],
                pseudo_target=pseudo_target,
                weight_map=weight_map,
                reliability_mask=reliability_mask,
            )
            if modality_mask is not None:
                loss_m = loss_m * modality_mask[:, m].mean()
            L_bac_kd = L_bac_kd + loss_m
        L_bac_kd = L_bac_kd / n_modalities

        # 6. CSC Loss
        L_csc = self.csc(logits_list, pseudo_target.detach(), reliability_mask=reliability_mask)

        # 7. Aggregation
        tau_sq = cfg.tau ** 2
        L_total = cfg.alpha * L_task + cfg.beta * tau_sq * L_bac_kd + cfg.gamma * L_csc

        info = {
            "L_total": L_total.item(),
            "L_task": L_task.item(),
            "L_BA_KD": L_bac_kd.item(),
            "L_CSC": L_csc.item(),
            "reliability_alpha": self.reliability_filter.get_alpha(epoch),
        }

        return L_total, info


---

## 12. Multi-Encoder 3D UNet and Fusion Model
Constructs the segmentation network. It has 4 separate 3D UNet branches (one per MRI modality) and a region-aware fusion module to combine multi-modal features at the final scale.
Includes stacked input handling for multi-GPU training with `nn.DataParallel` compatibility (DataParallel inputs are split and passed as a stacked tensor).

In [ ]:
class ConvBlock3D(nn.Module):
    """Two consecutive 3D Conv-InstanceNorm-ReLU layers."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DownBlock(nn.Module):
    """Downsample via MaxPool then ConvBlock."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.conv = ConvBlock3D(in_ch, out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(self.pool(x))


class UpBlock(nn.Module):
    """Upsample via trilinear interpolation, concat skip, then ConvBlock."""

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.conv = ConvBlock3D(in_ch + skip_ch, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class UNet3DBranch(nn.Module):
    """Full 3D UNet encoder-decoder for one modality."""

    def __init__(self, in_channels: int = 1, num_classes: int = 4, base_ch: int = 16):
        super().__init__()
        ch = [base_ch, base_ch * 2, base_ch * 4, base_ch * 8]

        self.enc0 = ConvBlock3D(in_channels, ch[0])
        self.enc1 = DownBlock(ch[0], ch[1])
        self.enc2 = DownBlock(ch[1], ch[2])
        self.bottleneck = DownBlock(ch[2], ch[3])

        self.dec2 = UpBlock(ch[3], ch[2], ch[2])
        self.dec1 = UpBlock(ch[2], ch[1], ch[1])
        self.dec0 = UpBlock(ch[1], ch[0], ch[0])

        self.head = nn.Conv3d(ch[0], num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        bn = self.bottleneck(e2)
        d2 = self.dec2(bn, e2)
        d1 = self.dec1(d2, e1)
        d0 = self.dec0(d1, e0)
        return self.head(d0)

    def forward_features(self, x: torch.Tensor):
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        bn = self.bottleneck(e2)
        d2 = self.dec2(bn, e2)
        d1 = self.dec1(d2, e1)
        d0 = self.dec0(d1, e0)
        logits = self.head(d0)
        return logits, d0


class RegionAwareFusion(nn.Module):
    """Fuse decoder features from all available modalities using region attention."""

    def __init__(self, feature_ch: int = 16, num_classes: int = 4, num_modalities: int = 4):
        super().__init__()
        self.num_classes = num_classes
        self.num_modalities = num_modalities

        self.attention = nn.Sequential(
            nn.Conv3d(feature_ch * num_modalities, feature_ch, 1, bias=False),
            nn.InstanceNorm3d(feature_ch, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv3d(feature_ch, num_modalities * num_classes, 1),
        )

        self.fuse_head = nn.Sequential(
            ConvBlock3D(feature_ch, feature_ch),
            nn.Conv3d(feature_ch, num_classes, kernel_size=1),
        )

    def forward(self, features_list: List[torch.Tensor]) -> torch.Tensor:
        B = features_list[0].shape[0]
        M = self.num_modalities
        C = self.num_classes
        spatial = features_list[0].shape[2:]

        concat = torch.cat(features_list, dim=1)
        attn_raw = self.attention(concat)
        attn = attn_raw.view(B, M, C, *spatial)
        attn = F.softmax(attn, dim=1)

        feats = torch.stack(features_list, dim=1)
        attn_avg = attn.mean(dim=2, keepdim=True)
        fused = (feats * attn_avg).sum(dim=1)

        return self.fuse_head(fused)


class MultiEncoderUNet(nn.Module):
    """Multi-encoder 3D UNet architecture."""

    def __init__(self, num_modalities: int = 4, num_classes: int = 4, base_ch: int = 16, use_fusion: bool = True):
        super().__init__()
        self.num_modalities = num_modalities
        self.num_classes = num_classes
        self.use_fusion = use_fusion

        self.branches = nn.ModuleList([
            UNet3DBranch(in_channels=1, num_classes=num_classes, base_ch=base_ch)
            for _ in range(num_modalities)
        ])

        if use_fusion:
            self.fusion = RegionAwareFusion(
                feature_ch=base_ch, num_classes=num_classes, num_modalities=num_modalities
            )

    def forward(self, inputs) -> List[torch.Tensor]:
        # Handle stacked tensor input for DataParallel compatibility
        if isinstance(inputs, torch.Tensor):
            inputs = [inputs[:, i] for i in range(self.num_modalities)]
        logits_list = []
        for branch, x in zip(self.branches, inputs):
            logits_list.append(branch(x))
        return logits_list

    def forward_with_fusion(self, inputs):
        # Handle stacked tensor input for DataParallel compatibility
        if isinstance(inputs, torch.Tensor):
            inputs = [inputs[:, i] for i in range(self.num_modalities)]
        logits_list = []
        features_list = []

        for branch, x in zip(self.branches, inputs):
            logits, feats = branch.forward_features(x)
            logits_list.append(logits)
            features_list.append(feats)

        fused_logits = None
        if self.use_fusion:
            fused_logits = self.fusion(features_list)

        return logits_list, fused_logits

# Creators
def create_rfnet(num_classes: int = 4) -> MultiEncoderUNet:
    return MultiEncoderUNet(num_modalities=4, num_classes=num_classes, base_ch=32, use_fusion=True)

def create_mmformer(num_classes: int = 4) -> MultiEncoderUNet:
    return MultiEncoderUNet(num_modalities=4, num_classes=num_classes, base_ch=40, use_fusion=True)

def create_lightweight(num_classes: int = 4) -> MultiEncoderUNet:
    return MultiEncoderUNet(num_modalities=4, num_classes=num_classes, base_ch=16, use_fusion=True)

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


---

## 13. BraTS and Dummy Dataset Pipeline
Provides loading utilities for real BraTS NIfTI files, as well as a synthetic Dummy dataset class designed for testing and running the training loop immediately without downloading large medical images.

In [ ]:
def _normalize_volume(vol: np.ndarray) -> np.ndarray:
    mask = vol > 0
    if mask.sum() == 0:
        return vol
    mean = vol[mask].mean()
    std = vol[mask].std() + 1e-8
    vol_norm = np.zeros_like(vol)
    vol_norm[mask] = (vol[mask] - mean) / std
    return vol_norm


def _crop_to_nonzero(volumes: List[np.ndarray], label: np.ndarray, margin: int = 5):
    combined = np.zeros_like(volumes[0])
    for v in volumes:
        combined += np.abs(v)

    nonzero = np.where(combined > 0)
    if len(nonzero[0]) == 0:
        return volumes, label

    d_min, d_max = nonzero[0].min(), nonzero[0].max()
    h_min, h_max = nonzero[1].min(), nonzero[1].max()
    w_min, w_max = nonzero[2].min(), nonzero[2].max()

    D, H, W = volumes[0].shape
    d_min = max(0, d_min - margin)
    d_max = min(D, d_max + margin + 1)
    h_min = max(0, h_min - margin)
    h_max = min(H, h_max + margin + 1)
    w_min = max(0, w_min - margin)
    w_max = min(W, w_max + margin + 1)

    cropped_vols = [v[d_min:d_max, h_min:h_max, w_min:w_max] for v in volumes]
    cropped_label = label[d_min:d_max, h_min:h_max, w_min:w_max]

    return cropped_vols, cropped_label


def _pad_or_crop_to_size(vol: np.ndarray, target_size: Tuple[int, int, int]) -> np.ndarray:
    D, H, W = vol.shape
    tD, tH, tW = target_size
    result = np.zeros((tD, tH, tW), dtype=vol.dtype)

    sd = max(0, (D - tD) // 2)
    sh = max(0, (H - tH) // 2)
    sw = max(0, (W - tW) // 2)

    td = max(0, (tD - D) // 2)
    th = max(0, (tH - H) // 2)
    tw = max(0, (tW - W) // 2)

    copy_d = min(D, tD)
    copy_h = min(H, tH)
    copy_w = min(W, tW)

    result[td:td+copy_d, th:th+copy_h, tw:tw+copy_w] = vol[sd:sd+copy_d, sh:sh+copy_h, sw:sw+copy_w]
    return result


class DummyBraTSDataset(Dataset):
    """Synthetic generator for rapid debugging and training execution."""

    def __init__(self, num_samples=20, crop_size=(64, 64, 64)):
        self.n = num_samples
        self.crop_size = crop_size

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        D, H, W = self.crop_size
        mods = [torch.randn(1, D, H, W) for _ in range(4)]
        label = torch.randint(0, 4, (D, H, W))
        return mods, label, f"dummy_{idx:04d}"


def dummy_collate(batch):
    mods_list = [[] for _ in range(4)]
    labels, names = [], []
    for m, l, n in batch:
        for i in range(4):
            mods_list[i].append(m[i])
        labels.append(l)
        names.append(n)
    return ([torch.stack(mods_list[i]) for i in range(4)], torch.stack(labels), names)


---

## 14. Segmentation Evaluation Metrics
Implements Dice Similarity Coefficient, Sensitivity (Recall), and 95th Percentile Hausdorff Distance (HD95) for the three standard BraTS composite regions:
* **Whole Tumor (WT):** Necrotic Core + Edema + Enhancing Tumor (classes 1 + 2 + 3).
* **Tumor Core (TC):** Necrotic Core + Enhancing Tumor (classes 1 + 3).
* **Enhancing Tumor (ET):** Enhancing Tumor only (class 3).

In [ ]:
def dice_score(pred: np.ndarray, target: np.ndarray) -> float:
    pred_sum = pred.sum()
    target_sum = target.sum()
    if pred_sum == 0 and target_sum == 0:
        return 1.0
    if pred_sum == 0 or target_sum == 0:
        return 0.0
    intersection = (pred & target).sum()
    return 2.0 * intersection / (pred_sum + target_sum)


def hausdorff_distance_95(pred: np.ndarray, target: np.ndarray) -> float:
    pred_sum = pred.sum()
    target_sum = target.sum()
    if pred_sum == 0 and target_sum == 0:
        return 0.0
    if pred_sum == 0 or target_sum == 0:
        return float(np.sqrt(sum(s ** 2 for s in pred.shape)))

    pred_border = pred ^ ndimage.binary_erosion(pred)
    target_border = target ^ ndimage.binary_erosion(target)

    dt_pred = ndimage.distance_transform_edt(~pred_border)
    dt_target = ndimage.distance_transform_edt(~target_border)

    d_pred_to_target = dt_target[pred_border]
    d_target_to_pred = dt_pred[target_border]

    if len(d_pred_to_target) == 0 or len(d_target_to_pred) == 0:
        return float(np.sqrt(sum(s ** 2 for s in pred.shape)))

    all_distances = np.concatenate([d_pred_to_target, d_target_to_pred])
    return float(np.percentile(all_distances, 95))


def sensitivity(pred: np.ndarray, target: np.ndarray) -> float:
    target_sum = target.sum()
    if target_sum == 0:
        return 1.0
    tp = (pred & target).sum()
    return float(tp) / float(target_sum)


def get_composite_regions(segmentation: np.ndarray) -> Dict[str, np.ndarray]:
    return {
        "WT": (segmentation > 0),
        "TC": np.isin(segmentation, [1, 3]),
        "ET": (segmentation == 3),
    }


def evaluate_subject(pred: np.ndarray, target: np.ndarray, compute_hd95: bool = True) -> Dict[str, float]:
    pred_regions = get_composite_regions(pred)
    target_regions = get_composite_regions(target)

    results = {}
    for region in ["WT", "TC", "ET"]:
        p = pred_regions[region]
        t = target_regions[region]
        results[f"Dice_{region}"] = dice_score(p, t)
        results[f"Sens_{region}"] = sensitivity(p, t)
        if compute_hd95:
            results[f"HD95_{region}"] = hausdorff_distance_95(p, t)

    return results


---

## 15. Training Loop
Functions to run one epoch of training, perform validation, and execute the full training schedule with weight resets.
Includes implementation of **Automatic Mixed Precision (AMP)** and optimization for multi-GPU training.

In [ ]:
def train_one_epoch(model, criterion, loader, optimizer, epoch, cfg, device, scaler=None):
    model.train()
    running = {"L_total": 0.0, "L_task": 0.0, "L_BA_KD": 0.0, "L_CSC": 0.0}
    n_batches = 0

    for batch in loader:
        modalities, labels = batch[0], batch[1]
        # non_blocking=True overlaps copy operations with host processing
        modalities = [m.to(device, non_blocking=True) for m in modalities]
        labels = labels.to(device, non_blocking=True)
        B = labels.shape[0]

        mod_mask = generate_modality_mask(B, cfg.num_modalities, cfg.mask_rate, device)

        masked_inputs = []
        for m_idx in range(cfg.num_modalities):
            scale = mod_mask[:, m_idx].view(B, 1, 1, 1, 1)
            masked_inputs.append(modalities[m_idx] * scale)

        # Stack inputs along dimension 1 for stable scatter in nn.DataParallel
        stacked_inputs = torch.stack(masked_inputs, dim=1)  # (B, M, C, D, H, W)

        optimizer.zero_grad(set_to_none=True)

        # Automatic Mixed Precision (AMP)
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits_list = model(stacked_inputs)
                loss, info = criterion(logits_list, labels, epoch, mod_mask)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits_list = model(stacked_inputs)
            loss, info = criterion(logits_list, labels, epoch, mod_mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        for k in running:
            running[k] += info.get(k, 0.0)
        n_batches += 1

    return {k: v / max(n_batches, 1) for k, v in running.items()}


@torch.no_grad()
def validate(model, loader, device, use_amp=False):
    model.eval()
    dice_sums = {"WT": 0.0, "TC": 0.0, "ET": 0.0}
    n_subjects = 0

    for batch in loader:
        modalities, labels = batch[0], batch[1]
        modalities = [m.to(device, non_blocking=True) for m in modalities]
        stacked_modalities = torch.stack(modalities, dim=1)
        labels_np = labels.numpy()

        if use_amp:
            with torch.cuda.amp.autocast():
                logits_list = model(stacked_modalities)
                avg_logits = torch.stack(logits_list).mean(dim=0)
                preds = avg_logits.argmax(dim=1).cpu().numpy()
        else:
            logits_list = model(stacked_modalities)
            avg_logits = torch.stack(logits_list).mean(dim=0)
            preds = avg_logits.argmax(dim=1).cpu().numpy()

        for i in range(preds.shape[0]):
            p, t = preds[i], labels_np[i]
            for region, fn in [("WT", lambda x: x > 0),
                               ("TC", lambda x: np.isin(x, [1, 3])),
                               ("ET", lambda x: x == 3)]:
                p_m, t_m = fn(p), fn(t)
                inter = (p_m & t_m).sum()
                union = p_m.sum() + t_m.sum()
                dice_sums[region] += (2.0 * inter / union) if union > 0 else 1.0
            n_subjects += 1

    if n_subjects == 0:
        return {"Dice_WT": 0, "Dice_TC": 0, "Dice_ET": 0}
    return {f"Dice_{r}": dice_sums[r] / n_subjects for r in ["WT", "TC", "ET"]}


---

## 16. Run Training Pipeline (Demonstration)
Runs a short lightweight training demonstration using the `DummyBraTSDataset` to verify the pipeline compiles and trains correctly.
Includes model compilation and data loader optimizations.

In [ ]:
# Training parameters for demonstration
DEMO_EPOCHS = 10
BATCH_SIZE = 2
LR = 1e-4

# Enable cuDNN benchmark auto-tuner
torch.backends.cudnn.benchmark = True

print("Initializing lightweight model, HAGSS config, and loss...")
cfg = HAGSSConfig(total_epochs=DEMO_EPOCHS, reload_interval=5)
model = create_lightweight(cfg.num_classes)

# 1. Compile Optimization (PyTorch 2.x)
if hasattr(torch, "compile"):
    try:
        model = torch.compile(model)
        print("Model compiled successfully using torch.compile")
    except Exception as e:
        print(f"Skipping torch.compile: {e}")

# 2. Multi-GPU DataParallel scaling
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with nn.DataParallel")
    model = nn.DataParallel(model)
model = model.to(device)

criterion = HAGSSLoss(cfg).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

# 3. Mixed Precision GradScaler
scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

print("Preparing dummy data loaders...")
# Configure workers based on OS to prevent Windows multiprocessing issues in notebooks
num_workers = min(4, os.cpu_count() or 2) if platform.system() != "Windows" else 0
train_ds = DummyBraTSDataset(num_samples=10, crop_size=(32, 32, 32))
val_ds = DummyBraTSDataset(num_samples=4, crop_size=(32, 32, 32))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=dummy_collate,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(num_workers > 0)
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    collate_fn=dummy_collate,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(num_workers > 0)
)

# Save initial weights for reload mechanism demonstration
raw_model = model.module if isinstance(model, nn.DataParallel) else model
if hasattr(raw_model, "_orig_mod"):
    raw_model = raw_model._orig_mod
initial_state = {k: v.clone() for k, v in raw_model.state_dict().items()}

print("\nStarting HAGSS Training Loop...")
for epoch in range(1, DEMO_EPOCHS + 1):
    t0 = time.time()
    
    # Reload Schedule Demo
    if epoch > 1 and (epoch - 1) % cfg.reload_interval == 0:
        raw_model.load_state_dict(initial_state)
        optimizer = optim.Adam(model.parameters(), lr=LR)
        print(f"\n*** Model Reloaded to initial weights at Epoch {epoch} ***")

    metrics = train_one_epoch(model, criterion, train_loader, optimizer, epoch, cfg, device, scaler=scaler)
    elapsed = time.time() - t0
    
    print(f"Epoch {epoch:>2d}/{DEMO_EPOCHS} | L_total={metrics['L_total']:.4f} | L_task={metrics['L_task']:.4f} | L_KD={metrics['L_BA_KD']:.6f} | L_CSC={metrics['L_CSC']:.6f} | ({elapsed:.1f}s)")
    
    if epoch % 5 == 0 or epoch == DEMO_EPOCHS:
        val = validate(model, val_loader, device, use_amp=(scaler is not None))
        print(f"  --> VAL Metrics: Dice_WT={val['Dice_WT']:.4f} | Dice_TC={val['Dice_TC']:.4f} | Dice_ET={val['Dice_ET']:.4f}")


---

## 17. Unit Tests & Mathematical Verification
Runs tests verifying the mathematical properties of HAGSS components (Logit Standardization, Sobel boundaries, override conditional logic, reliability decays).

In [ ]:
# Helper synthetic definitions for tests
B, C, D, H, W = 1, 4, 4, 4, 4
M = 4

def check(name, condition, detail=""):
    if condition:
        print(f"  [PASS] {name}")
    else:
        print(f"  [FAIL] {name} {detail}")
        raise AssertionError(f"Test '{name}' failed.")

print("Running Unit Tests...\n")

# 1. Logit Standardization Test
logit_std = LogitStandardization()
x = torch.randn(B, C, D, H, W, requires_grad=True)
y = logit_std(x)
check("LogitStandardization: shape matches", y.shape == x.shape)
check("LogitStandardization: mean is 0", y.mean(dim=1).abs().max().item() < 1e-4)
check("LogitStandardization: std is 1", (y.std(dim=1) - 1.0).abs().max().item() < 0.05)

# 2. Entropy Confidence Test
ent_conf = EntropyConfidence(num_classes=C)
entropy, confidence = ent_conf(x)
check("EntropyConfidence: entropy range check", (entropy >= 0).all() and (entropy <= math.log(C) + 1e-4).all())
check("EntropyConfidence: confidence in [0,1]", (confidence >= 0).all() and (confidence <= 1.0).all())

# 3. Sobel Boundary Test
sobel_3d = SobelBoundary3D(num_classes=C, threshold=0.5)
labels = torch.randint(0, C, (B, D, H, W))
boundary = sobel_3d(labels)
check("SobelBoundary3D: shape matches", boundary.shape == (B, 1, D, H, W))
check("SobelBoundary3D: mask is binary", set(boundary.unique().tolist()).issubset({0.0, 1.0}))

# 4. Reliability Filter Test
rel_filter = ReliabilityFilter(alpha_0=0.08, total_epochs=100)
check("ReliabilityFilter: alpha_0 initialization", abs(rel_filter.get_alpha(0) - 0.08) < 1e-6)
check("ReliabilityFilter: alpha decays to 0", abs(rel_filter.get_alpha(100) - 0.0) < 1e-6)

# 5. Group Formation Test
hagf = HierarchicalAdaptiveGroupFormation(num_classes=C, num_modalities=M)
logits_list = [torch.randn(B, C, D, H, W) for _ in range(M)]
pseudo = hagf(logits_list)
check("HAGF: pseudo-target shape matches", pseudo.shape == (B, C, D, H, W))

# 6. Boundary Aware Calibration Test
bac = BoundaryAwareCalibration(num_classes=C, lambda_b=2.0, lambda_u=1.0)
wmap = bac.compute_weight_map(labels, logits_list)
check("BAC: weight map range constraints", (wmap >= 1.0).all() and (wmap <= 4.0 + 1e-2).all())

# 7. Modality Masking Test
mask = generate_modality_mask(batch_size=10, num_modalities=4, mask_rate=0.2)
check("generate_modality_mask: shape constraints", mask.shape == (10, 4))
check("generate_modality_mask: active modality exists", (mask.sum(dim=1) >= 1).all())

print("\nAll HAGSS components verified successfully!")
